# Python — Threading, Multiprocessing, and asyncio
**Day 9 — Python Concurrency**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Python has three concurrency models and they solve different problems. Threading is for I/O-bound tasks — waiting for network, disk, API. Multiprocessing is for CPU-bound tasks — computation, data processing. asyncio is for highly concurrent I/O with many connections. <strong>The GIL is the key:</strong> threading doesn't give you parallelism for CPU work, but it does work for I/O waits.
</div>

### Why It Matters
In data pipelines, network latency (calling APIs, waiting for database queries) is frequently the primary bottleneck. Using the right concurrency model turns a 2-hour pipeline into a 5-minute pipeline. Using the wrong one either breaks your application or provides zero speedup due to the Global Interpreter Lock (GIL).

## Concurrency Models Reference Card

```text
Model          | Parallelism | Best For           | GIL Impact
---------------|-------------|--------------------|-----------
threading      | Concurrent  | I/O-bound (API)    | GIL blocks CPU work
multiprocessing| Parallel    | CPU-bound (compute)| Separate process each
asyncio        | Concurrent  | Many I/O connects  | Single thread, event loop
```

```python
# Threading — use for parallel I/O (APIs, DB queries)
from concurrent.futures import ThreadPoolExecutor
# → ThreadPoolExecutor(max_workers=10)

# Multiprocessing — use for parallel CPU work (pandas, heavy math)
from concurrent.futures import ProcessPoolExecutor
# → ProcessPoolExecutor(max_workers=4)

# asyncio — use for massive I/O concurrency (websockets, 1000s of APIs)
import asyncio
# → asyncio.run(main())
```

In [1]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_server_status(server_id: str) -> dict:
    """Simulate an API call — I/O bound."""
    time.sleep(0.1)   # network latency simulation
    return {"server_id": server_id, "status": "ok", "cpu": 72.5}

SERVER_IDS = [f"srv-{i:02d}" for i in range(1, 11)]   # 10 servers

# Sequential: 10 × 0.1s = ~1.0s
print("Starting Sequential Fetch...")
start = time.perf_counter()
results_seq = [fetch_server_status(sid) for sid in SERVER_IDS]
print(f"Sequential: {time.perf_counter() - start:.2f}s")

# ThreadPoolExecutor: concurrent I/O — all 10 run simultaneously
print("\nStarting ThreadPoolExecutor Fetch...")
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as executor:
    results_thread = list(executor.map(fetch_server_status, SERVER_IDS))
print(f"ThreadPool: {time.perf_counter() - start:.2f}s")
print(f"Results: {len(results_thread)} servers polled")

Starting Sequential Fetch...
Sequential: 1.01s

Starting ThreadPoolExecutor Fetch...
ThreadPool: 0.11s
Results: 10 servers polled


## Execution Mechanics & The GIL

Python executes code using a single main thread containing the **Global Interpreter Lock (GIL)**.

**Execution model:**
1. **Threading:** Multiple threads exist, but only one executes Python bytecode at a time. When a thread hits an I/O operation (like an HTTP call), it releases the GIL. Another thread can then run. This provides *concurrency*, not *parallelism*.
2. **Multiprocessing:** Python spawns entirely separate OS processes. Each process gets its own memory space and its own GIL. This provides true *parallelism* for CPU-bound tasks, bypassing the GIL entirely.
3. **asyncio:** A single thread runs an Event Loop. Coroutines completely execute in this single thread. When a coroutine awaits an I/O operation (like `await aiohttp.get()`), it explicitly yields control back to the event loop, which seamlessly switches to another task.

In [2]:
from concurrent.futures import ProcessPoolExecutor
import time
import numpy as np

def compute_rolling_stats(server_data: tuple) -> dict:
    """CPU-bound: compute statistics for a server's time series."""
    server_id, readings = server_data
    arr = np.array(readings)
    return {
        "server_id": server_id,
        "mean": float(arr.mean()),
        "p95": float(np.percentile(arr, 95)),
        "std": float(arr.std()),
    }

# Generate sample Citi-adjacent data
server_datasets = [
    (f"srv-{i:02d}", list(np.random.uniform(40, 100, 100000)))
    for i in range(1, 9)
]

# Sequential computation
print("Starting Sequential CPU computation...")
start = time.perf_counter()
results_seq = [compute_rolling_stats(d) for d in server_datasets]
print(f"Sequential: {time.perf_counter() - start:.3f}s")

# ProcessPoolExecutor — utilizes multiple CPU cores naturally bypassing the GIL
print("\nStarting ProcessPool CPU computation...")
start = time.perf_counter()
# Note: In a real script, this Must be inside `if __name__ == '__main__':`
with ProcessPoolExecutor(max_workers=4) as executor:
    results_proc = list(executor.map(compute_rolling_stats, server_datasets))
print(f"ProcessPool: {time.perf_counter() - start:.3f}s")

Starting Sequential CPU computation...
Sequential: 0.125s

Starting ProcessPool CPU computation...
ProcessPool: 0.045s


In [3]:
# Pipeline showcase: Concurrent I/O with asyncio
import asyncio
import time

async def async_fetch_metrics(server_id: str) -> dict:
    """Async version — explicitly yields control during I/O wait."""
    await asyncio.sleep(0.1)   # simulates network wait without blocking threads
    return {"server_id": server_id, "cpu": 72.5}

async def fetch_all_servers(server_ids: list[str]) -> list[dict]:
    """Run all fetches concurrently."""
    tasks = [async_fetch_metrics(sid) for sid in server_ids]
    return await asyncio.gather(*tasks)    # all coroutines run concurrently

# Run the event loop against the simulation
SERVER_IDS = [f"srv-db-{i:02d}" for i in range(1, 21)]   # 20 servers

print(f"Starting asyncio fetch for {len(SERVER_IDS)} servers...")
start = time.perf_counter()

# Note: in Jupyter await works directly. In standard Python script: asyncio.run(fetch_all_servers(...))
results_async = await fetch_all_servers(SERVER_IDS)

print(f"asyncio ({len(results_async)} servers): {time.perf_counter() - start:.2f}s")
print(f"Sample result: {results_async[0]}")

Starting asyncio fetch for 20 servers...
asyncio (20 servers): 0.11s
Sample result: {'server_id': 'srv-db-01', 'cpu': 72.5}


## Advanced Concurrency Tools

The standard libraries `concurrent.futures`, `threading`, and `asyncio` offer comprehensive primitives to safely synchronize data and handle concurrency limits out of the box.

In [4]:
import asyncio
import time
import threading
from concurrent.futures import ThreadPoolExecutor

# 1. asyncio.Semaphore — limit maximum concurrency
async def limited_api_call(sem, task_id):
    async with sem:  # Only N tasks can enter this block concurrently
        await asyncio.sleep(0.05)
        return f"API Response: {task_id}"

# 2. threading.Lock — safely modify shared state across threads
counter_state = 0
state_lock = threading.Lock()

def increment_safe():
    global counter_state
    with state_lock:         # GIL is not enough for complex thread synchronizations!
        current = counter_state
        time.sleep(0.001)    # simulate small delay
        counter_state = current + 1

async def showcase_tools():
    print("Testing Semaphore (max 2 concurrent at a time)...")
    sem = asyncio.Semaphore(2)  # Cap at 2 concurrent limits
    tasks = [limited_api_call(sem, i) for i in range(4)]
    results = await asyncio.gather(*tasks)
    print("Semaphore results:", results)
    
    print("\nTesting Thread Safety with locks...")
    with ThreadPoolExecutor(max_workers=10) as ex:
        ex.map(lambda _: increment_safe(), range(10))
    print("Safe synchronized count value:", counter_state)

await showcase_tools()

Testing Semaphore (max 2 concurrent at a time)...
Semaphore results: ['API Response: 0', 'API Response: 1', 'API Response: 2', 'API Response: 3']

Testing Thread Safety with locks...
Safe synchronized count value: 10


## Interview Q&A

**Q: What is the GIL and why does it matter?**<br>
A: The Global Interpreter Lock is Python's mutex that allows only one thread to execute Python bytecode at a time. Threading doesn't give you parallelism for CPU-bound work. If you need CPU parallelism, use multiprocessing since it leverages separate OS processes (each with a separate GIL).

**Q: When would you use threading vs multiprocessing in a data pipeline?**<br>
A: Threading is ideal for fetching data from many APIs or databases concurrently (I/O bound). Multiprocessing is required for data transformation, feature engineering, or heavy statistical computation on large DataFrames (CPU bound).

**Q: What is `asyncio.gather()`?**<br>
A: It runs multiple coroutines concurrently within the same single-threaded event loop. It returns only when all coroutines successfully complete and their results are aggregated in a single list. It operates equivalently to `Promise.all()` in JavaScript.

**Q: What is `concurrent.futures` and why prefer it over raw threading/multiprocessing?**<br>
A: It's a high-level modern interface providing `ThreadPoolExecutor` and `ProcessPoolExecutor`. They share the exact same API, meaning you can trivially swap execution models. It's significantly cleaner and less error-prone than manually managing thread spawning, joining, and queueing.

**Q: Design a continuous pipeline to ingest JSON from 1000 slow API endpoints and perform heavy mathematical aggregations on the payload.**<br>
A: I would deploy an asynchronous approach (`asyncio` and `aiohttp`) to concurrently fetch payloads from the 1000 APIs to gracefully handle the massive I/O wait times. Once the raw payloads are dynamically aggregated in memory, I would dispatch the mathematical aggregations securely to a `ProcessPoolExecutor` to bypass the GIL and recruit all logical CPU cores.

## The Citi Angle

**Managing Hybrid Workloads in Telemetry Analytics**

When polling thousands of enterprise servers across global environments, combining massive I/O concurrency with raw CPU parallelism is an absolute requirement.

```python
import asyncio
from concurrent.futures import ProcessPoolExecutor

async def poll_servers_async(server_list: list) -> list[dict]:
    # Phase 1: Heavy I/O bound - Use asyncio to hit endpoints without thread blocking
    fetch_tasks = [http_get_async(f"http://{srv}/metrics") for srv in server_list]
    payloads = await asyncio.gather(*fetch_tasks)
    return payloads

def process_metrics_pipeline(payloads: list[dict]):
    # Phase 2: CPU bound - Use ProcessPool to rip through giant computational dictionaries
    with ProcessPoolExecutor() as executor:
        enriched_results = list(executor.map(cpu_intensive_enrichment, payloads))
    return enriched_results
```

*"In our aggressive monitoring pipeline for the global trading desks, we were fetching 2,000 server metrics per minute. Using basic threading succeeded for the initial fetches, but the pipeline repeatedly stalled during the complex payload aggregations owing to the GIL backing up. We completely redesigned the architecture: asyncio powered the network fetches to capture everything practically instantly, and we piped the resulting payloads cleanly into a ProcessPoolExecutor. We effectively hit perfect parallel CPU utilization, instantly plummeting end-to-end latency from 45 seconds down to under 2 seconds."*